#  Exploratory Data Analysis: Climate-Socioeconomic Panel (1900–2023)

**Objective**: Understand data structure, target behavior, feature relationships, and inform preprocessing/feature engineering decisions.

**Dataset**: 195 countries × 124 years ≈ 100K rows, 26 columns  
**Targets**: `Temperature_Anomaly`, `CO2_Emissions`  
**Key Questions**:
1. Is the panel balanced? Any missing country-year combinations?
2. How have global temperature and CO₂ trends evolved? Any structural breaks?
3. Which socioeconomic factors correlate most strongly with climate targets?
4. Are there multicollinearity concerns among predictors?
5. What features should we engineer (lags, ratios, interactions)?

*Note: All visualizations saved to `../outputs/figures/eda_*` for paper inclusion.*

In [10]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import stats
# from statsmodels.tsa.stattools import adfuller
# from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Plotting config
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 150
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paths
DATA_PATH = '../data/raw/global_warming_dataset.csv'
OUTPUT_FIG = '../outputs/figures/eda_'

print("Environment ready")

Environment ready


In [11]:
# Load the dataset
df = pd.read_csv(DATA_PATH)
print("First 5 rows:")
display(df.head())
print("\n Columns and Data Types:")
print(df.dtypes)
print(f"\ Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n Statistical Summary (numeric columns):")
display(df.describe(include='number').T)
print("\n Missing Values per Column:")
print(df.isnull().sum())

print("\nUnique Values in 'Country' and 'Year':")
print(f"   Countries: {df['Country'].nunique()}")
print(f"   Years: {df['Year'].nunique()} | Range: {df['Year'].min()} – {df['Year'].max()}")

First 5 rows:


,Country,Year,Temperature_Anomaly,CO2_Emissions,Population,Forest_Area,GDP,Renewable_Energy_Usage,Methane_Emissions,Sea_Level_Rise,...,Waste_Management,Per_Capita_Emissions,Industrial_Activity,Air_Pollution_Index,Biodiversity_Index,Ocean_Acidification,Fossil_Fuel_Usage,Energy_Consumption_Per_Capita,Policy_Score,Average_Temperature
0,Country_103,1913,-1.163537,8.876061e+08,1.627978e+08,54.872178,6.139887e+12,76.710013,8.317626e+06,8.111839,...,82.691409,2.285351,4.060975,150.285539,90.073356,8.025470,39.163860,1480.164332,78.870012,20.825292
1,Country_180,1950,-0.432122,4.497517e+08,4.281359e+08,84.051006,2.601447e+12,68.450021,6.206540e+06,42.025915,...,59.322883,17.411668,85.300604,27.305922,88.289837,8.021719,28.252554,1482.730048,32.600905,28.720587
2,Country_93,2014,0.444954,4.579080e+08,4.926732e+08,72.295357,5.192677e+12,36.725699,1.056885e+06,20.953840,...,94.982931,12.039703,83.804880,216.911429,86.936256,7.647408,61.548382,706.918809,37.671300,15.014084
3,Country_15,2020,-1.171616,5.049503e+08,1.252169e+09,17.259684,8.252128e+12,77.547901,1.986813e+06,45.599595,...,62.064250,2.853957,47.014265,35.869182,44.904331,7.569353,82.423750,2616.238324,86.581725,-1.277086
4,Country_107,1964,-0.564038,6.898891e+08,2.932960e+08,44.438605,8.560746e+12,10.019576,3.313252e+06,7.652150,...,84.431279,19.801173,89.379613,284.263093,8.102916,8.015415,29.964450,4975.683780,20.618406,2.861989



 Columns and Data Types:
Country                           object
Year                               int64
Temperature_Anomaly              float64
CO2_Emissions                    float64
Population                       float64
Forest_Area                      float64
GDP                              float64
Renewable_Energy_Usage           float64
Methane_Emissions                float64
Sea_Level_Rise                   float64
Arctic_Ice_Extent                float64
Urbanization                     float64
Deforestation_Rate               float64
Extreme_Weather_Events             int64
Average_Rainfall                 float64
Solar_Energy_Potential           float64
Waste_Management                 float64
Per_Capita_Emissions             float64
Industrial_Activity              float64
Air_Pollution_Index              float64
Biodiversity_Index               float64
Ocean_Acidification              float64
Fossil_Fuel_Usage                float64
Energy_Consumption_Per_Capita  

,count,mean,std,min,25%,50%,75%,max
Year,100000.0,1.961422e+03,3.575880e+01,1.900000e+03,1.930000e+03,1.961000e+03,1.992000e+03,2.023000e+03
Temperature_Anomaly,100000.0,2.647292e-03,1.154579e+00,-1.999981e+00,-9.994490e-01,3.523434e-03,9.980670e-01,1.999958e+00
CO2_Emissions,100000.0,5.007007e+08,2.887388e+08,1.022271e+05,2.504223e+08,5.006769e+08,7.506318e+08,9.999793e+08
Population,100000.0,7.501973e+08,4.328709e+08,1.013054e+06,3.762545e+08,7.499212e+08,1.125222e+09,1.499992e+09
Forest_Area,100000.0,5.001241e+01,2.880629e+01,5.188446e-05,2.496040e+01,5.014126e+01,7.489830e+01,9.999983e+01
GDP,100000.0,4.995923e+12,2.883117e+12,1.524737e+08,2.507503e+12,4.985849e+12,7.489082e+12,9.999894e+12
Renewable_Energy_Usage,100000.0,5.011566e+01,2.884149e+01,5.306081e-03,2.530224e+01,5.001158e+01,7.509594e+01,9.999949e+01
Methane_Emissions,100000.0,5.003713e+06,2.887606e+06,1.133254e+03,2.501312e+06,4.997724e+06,7.507855e+06,9.999928e+06
Sea_Level_Rise,100000.0,2.256805e+01,1.590816e+01,-4.999954e+00,8.739127e+00,2.272842e+01,3.633315e+01,4.999768e+01
Arctic_Ice_Extent,100000.0,8.003934e+00,4.039130e+00,1.000036e+00,4.504867e+00,7.999331e+00,1.149950e+01,1.499988e+01



 Missing Values per Column:
Country                          0
Year                             0
Temperature_Anomaly              0
CO2_Emissions                    0
Population                       0
Forest_Area                      0
GDP                              0
Renewable_Energy_Usage           0
Methane_Emissions                0
Sea_Level_Rise                   0
Arctic_Ice_Extent                0
Urbanization                     0
Deforestation_Rate               0
Extreme_Weather_Events           0
Average_Rainfall                 0
Solar_Energy_Potential           0
Waste_Management                 0
Per_Capita_Emissions             0
Industrial_Activity              0
Air_Pollution_Index              0
Biodiversity_Index               0
Ocean_Acidification              0
Fossil_Fuel_Usage                0
Energy_Consumption_Per_Capita    0
Policy_Score                     0
Average_Temperature              0
dtype: int64

Unique Values in 'Country' and 'Year':
   Count

## Data Integrity & Panel Balance Check

In [12]:
# === DATA INTEGRITY CHECKS ===

# 1. Check for duplicate (Country, Year) pairs
duplicates = df.duplicated(subset=['Country', 'Year']).sum()
print(f"🔄 Duplicate (Country, Year) entries: {duplicates}")

# If duplicates exist, show example
if duplicates > 0:
    print("\nExample duplicates:")
    display(df[df.duplicated(subset=['Country', 'Year'], keep=False)].head())

# 2. Verify panel completeness: expected rows = 195 countries × 124 years = 24,180
expected_rows = 195 * 124
actual_rows = len(df)
print(f"\n Panel Completeness:")
print(f"   Expected rows (full panel): {expected_rows:,}")
print(f"   Actual rows: {actual_rows:,}")
print(f"   Coverage: {actual_rows / expected_rows * 100:.1f}%")

# 3. Records per country: is the panel balanced?
records_per_country = df.groupby('Country').size().describe()
print(f"\nRecords per Country (summary):")
print(records_per_country)

# 4. Identify countries with incomplete time series
incomplete = df.groupby('Country')['Year'].apply(lambda x: len(x) < 124)
n_incomplete = incomplete.sum()
print(f"\nCountries with incomplete time series (<124 years): {n_incomplete}")

# If any incomplete, list top 10 with fewest records
if n_incomplete > 0:
    print("\nTop 10 countries with fewest records:")
    print(df.groupby('Country').size().sort_values().head(10))

# 5. Quick check: any years missing globally?
all_years = set(range(1900, 2024))
present_years = set(df['Year'].unique())
missing_years = all_years - present_years
print(f"\nMissing years in dataset: {sorted(missing_years) if missing_years else 'None'}")

🔄 Duplicate (Country, Year) entries: 76203

Example duplicates:


,Country,Year,Temperature_Anomaly,CO2_Emissions,Population,Forest_Area,GDP,Renewable_Energy_Usage,Methane_Emissions,Sea_Level_Rise,...,Waste_Management,Per_Capita_Emissions,Industrial_Activity,Air_Pollution_Index,Biodiversity_Index,Ocean_Acidification,Fossil_Fuel_Usage,Energy_Consumption_Per_Capita,Policy_Score,Average_Temperature
0,Country_103,1913,-1.163537,8.876061e+08,1.627978e+08,54.872178,6.139887e+12,76.710013,8.317626e+06,8.111839,...,82.691409,2.285351,4.060975,150.285539,90.073356,8.025470,39.163860,1480.164332,78.870012,20.825292
1,Country_180,1950,-0.432122,4.497517e+08,4.281359e+08,84.051006,2.601447e+12,68.450021,6.206540e+06,42.025915,...,59.322883,17.411668,85.300604,27.305922,88.289837,8.021719,28.252554,1482.730048,32.600905,28.720587
2,Country_93,2014,0.444954,4.579080e+08,4.926732e+08,72.295357,5.192677e+12,36.725699,1.056885e+06,20.953840,...,94.982931,12.039703,83.804880,216.911429,86.936256,7.647408,61.548382,706.918809,37.671300,15.014084
3,Country_15,2020,-1.171616,5.049503e+08,1.252169e+09,17.259684,8.252128e+12,77.547901,1.986813e+06,45.599595,...,62.064250,2.853957,47.014265,35.869182,44.904331,7.569353,82.423750,2616.238324,86.581725,-1.277086
4,Country_107,1964,-0.564038,6.898891e+08,2.932960e+08,44.438605,8.560746e+12,10.019576,3.313252e+06,7.652150,...,84.431279,19.801173,89.379613,284.263093,8.102916,8.015415,29.964450,4975.683780,20.618406,2.861989



 Panel Completeness:
   Expected rows (full panel): 24,180
   Actual rows: 100,000
   Coverage: 413.6%

Records per Country (summary):
count    195.000000
mean     512.820513
std       21.031973
min      457.000000
25%      498.000000
50%      516.000000
75%      528.000000
max      557.000000
dtype: float64

Countries with incomplete time series (<124 years): 0

Missing years in dataset: None


In [13]:
# === INVESTIGATE: Why ~4-5 records per Country-Year? ===

# 1. Check how many records exist per (Country, Year) combination
records_per_group = df.groupby(['Country', 'Year']).size().reset_index(name='count')
print("🔍 Records per (Country, Year) pair:")
print(records_per_group['count'].describe())

# 2. Show the distribution of counts
print(f"\n📊 Frequency of record counts per group:")
print(records_per_group['count'].value_counts().sort_index())

# 3. Pick one example country-year to inspect raw structure
example = df[(df['Country'] == df['Country'].iloc[0]) & (df['Year'] == 1900)]
print(f"\n📋 Example: {example['Country'].iloc[0]} in 1900 ({len(example)} records):")
display(example.T)  # Transpose to see all columns vertically

# 4. Check if any column varies within the same Country-Year
# (This reveals the hidden grouping factor)
print(f"\n🔎 Columns that vary within same (Country, Year):")
varying_cols = []
for col in df.columns:
    if col not in ['Country', 'Year']:
        # For each country-year, check if column has >1 unique value
        n_unique = df.groupby(['Country', 'Year'])[col].nunique().max()
        if n_unique > 1:
            varying_cols.append((col, n_unique))

# Show top varying columns
if varying_cols:
    varying_df = pd.DataFrame(varying_cols, columns=['Column', 'Max_Unique_Values'])
    print(varying_df.sort_values('Max_Unique_Values', ascending=False).head(10))
else:
    print("⚠️ No columns vary within Country-Year → duplicates may be exact copies")

# 5. Decision point: Should we aggregate to one row per Country-Year?
print(f"\n✅ Recommendation:")
if not varying_cols:
    print("→ All columns are identical within Country-Year → drop exact duplicates")
    print("   Code: df = df.drop_duplicates(subset=['Country', 'Year'])")
else:
    print("→ Some columns vary within Country-Year → investigate grouping logic")
    print("   Options: (a) aggregate with mean/sum, (b) keep as-is if representing scenarios/regions")

🔍 Records per (Country, Year) pair:
count    23797.000000
mean         4.202210
std          1.992073
min          1.000000
25%          3.000000
50%          4.000000
75%          5.000000
max         15.000000
Name: count, dtype: float64

📊 Frequency of record counts per group:
count
1     1634
2     3355
3     4527
4     4585
5     3934
6     2718
7     1624
8      816
9      345
10     153
11      63
12      31
13       8
14       3
15       1
Name: count, dtype: int64

📋 Example: Country_103 in 1900 (2 records):


,27530,43842
Country,Country_103,Country_103
Year,1900,1900
Temperature_Anomaly,1.816477,-1.388659
CO2_Emissions,246239311.066695,747080397.35253
Population,409107208.610274,286304784.250186
Forest_Area,26.230493,78.635837
GDP,5026039425292.311523,2104730330583.673096
Renewable_Energy_Usage,41.692163,63.011642
Methane_Emissions,4403641.717253,1054567.406578
Sea_Level_Rise,10.782809,31.553886



🔎 Columns that vary within same (Country, Year):
                   Column  Max_Unique_Values
0     Temperature_Anomaly                 15
1           CO2_Emissions                 15
2              Population                 15
3             Forest_Area                 15
4                     GDP                 15
5  Renewable_Energy_Usage                 15
6       Methane_Emissions                 15
7          Sea_Level_Rise                 15
8       Arctic_Ice_Extent                 15
9            Urbanization                 15

✅ Recommendation:
→ Some columns vary within Country-Year → investigate grouping logic
   Options: (a) aggregate with mean/sum, (b) keep as-is if representing scenarios/regions


In [ ]:
# === AGGREGATION: Create single row per (Country, Year) ===
id_cols = ['Country', 'Year']

# Select only numeric columns EXCLUDING the ID columns to prevent reset_index conflict
agg_cols = df.select_dtypes(include='number').columns.difference(id_cols)

# Aggregate using mean for all numeric features/targets
df_aggregated = df.groupby(id_cols)[agg_cols].mean().reset_index()

# Verify aggregation result
print("Post-aggregation dataset info:")
print(f"Shape: {df_aggregated.shape}")
print(f"Expected: 195 countries x 124 years = 24,180 rows")
print(f"Actual rows: {len(df_aggregated):,}")
print(f"Duplicate check: {df_aggregated.duplicated(subset=id_cols).sum()} duplicates")

# Verify targets are now single-valued per group
print("\nVerification: Unique values per (Country, Year) after aggregation:")
for col in ['Temperature_Anomaly', 'CO2_Emissions', 'GDP', 'Population']:
    n_unique = df_aggregated.groupby(id_cols)[col].nunique().max()
    print(f"  {col}: max {n_unique} unique value(s) per group")

# Update working dataframe
df = df_aggregated.copy()
print("\nAggregation complete. Proceeding with EDA on aggregated panel.")

ValueError: cannot insert Year, already exists